## Imports

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

In [2]:
import zipfile
# with zipfile.ZipFile('../data/raw2/Archive.zip', 'r') as zip_ref:
#     zip_ref.extractall('../data/raw2/')

## Load the dataset

In [3]:
df_orders = pd.read_csv('../data/raw/orders.csv')

In [4]:
# Reading data
df_orders_products_train = pd.read_csv('../data/raw/order_products_train.csv')

In [5]:
df_products = pd.read_csv("../data/raw/products.csv")

In [6]:
df_departments = pd.read_csv('../data/processed/departments_t.csv')

In [7]:
df_aisles = pd.read_csv('../data/raw/aisles.csv')

In [8]:
df_customers = pd.read_csv('../data/raw/customers.csv')

In [9]:
df_states = pd.read_csv('../data/raw/states.csv')

## Exploring the data

### orders

In [10]:
print(df_orders.columns)
print(df_orders.shape)
df_orders.head()

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')
(3421083, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


In [11]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 198.9 MB


In [12]:
df_orders[['order_number', 'order_dow','order_hour_of_day', 'days_since_prior_order']].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,"3,421,083.00","3,421,083.00","3,421,083.00","3,214,874.00"
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [13]:
# missing values
df_orders.isna().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

#### Analytical questions for`orders`

In [14]:
# Exploring the eval_set distribution
df_orders['eval_set'].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

In [15]:
# How many unique orders exist?
df_orders["order_id"].nunique()

3421083

In [16]:
# How many distinct customers are represented in the dataset?
df_orders["user_id"].nunique()

206209

In [17]:
# Orders by day of week
df_orders["order_dow"].value_counts().sort_index()

order_dow
0    600905
1    587478
2    467260
3    436972
4    426339
5    453368
6    448761
Name: count, dtype: int64

In [18]:
# Orders by hour of day
df_orders["order_hour_of_day"].value_counts().sort_index()

order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64

In [19]:
# Understanding missing values in days_since_prior_order
df_orders['days_since_prior_order'].isna().sum()

np.int64(206209)

In [20]:
# Logical check: first orders and null values
df_orders.loc[df_orders["order_number"] == 1, "days_since_prior_order"].isna().all()

np.True_

In [21]:
# Is order_id unique?
df_orders["order_id"].is_unique

True

In [22]:
# Is order_dow in the correct range?
df_orders["order_dow"].between(0, 6).all()

np.True_

In [23]:
# Is order_hour_of_day in the correct range?
df_orders["order_hour_of_day"].between(0, 23).all()

np.True_

#### Filtering the `orders` table to train orders

In [24]:
df_orders_train = df_orders[df_orders["eval_set"] == "train"]
df_orders_train.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.00
25,1492625,2,train,15,1,11,30.00
49,2196797,5,train,5,0,11,6.00
74,525192,7,train,21,2,11,6.00
78,880375,8,train,4,1,14,10.00


In [25]:
df_orders_train.drop(columns=["eval_set"], inplace=True)
df_orders_train.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,11,4,8,14.00
25,1492625,2,15,1,11,30.00
49,2196797,5,5,0,11,6.00
74,525192,7,21,2,11,6.00
78,880375,8,4,1,14,10.00


In [26]:
## Delete df_orders to free up memory
del df_orders

### order_products_train

In [27]:
print(df_orders_products_train.columns)
print(df_orders_products_train.shape)
df_orders_products_train.head()

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
(1384617, 4)


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [28]:
df_orders_products_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   order_id           1384617 non-null  int64
 1   product_id         1384617 non-null  int64
 2   add_to_cart_order  1384617 non-null  int64
 3   reordered          1384617 non-null  int64
dtypes: int64(4)
memory usage: 42.3 MB


In [29]:
df_orders_products_train.describe()

,order_id,product_id,add_to_cart_order,reordered
count,"1,384,617.00","1,384,617.00","1,384,617.00","1,384,617.00"
mean,"1,706,297.62","25,556.24",8.76,0.60
std,"989,732.65","14,121.27",7.42,0.49
min,1.00,1.00,1.00,0.00
25%,"843,370.00","13,380.00",3.00,0.00
50%,"1,701,880.00","25,298.00",7.00,1.00
75%,"2,568,023.00","37,940.00",12.00,1.00
max,"3,421,070.00","49,688.00",80.00,1.00


In [30]:
df_orders_products_train.isna().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

#### Analytical questions for `order_products_train`

In [31]:
# How many line items exist?
len(df_orders_products_train)

1384617

In [32]:
# How many unique orders are represented in this table?
df_orders_products_train["order_id"].nunique()

131209

In [33]:
# How many unique products are represented in this table?
df_orders_products_train["product_id"].nunique()

39123

In [34]:
# Exploring reorder behavior
df_orders_products_train["reordered"].value_counts(dropna=False)
df_orders_products_train["reordered"].value_counts(normalize=True)  # Reorder share

reordered
1   0.60
0   0.40
Name: proportion, dtype: float64

In [35]:

# Understanding add_to_cart_order
df_orders_products_train["add_to_cart_order"].describe()

count   1,384,617.00
mean            8.76
std             7.42
min             1.00
25%             3.00
50%             7.00
75%            12.00
max            80.00
Name: add_to_cart_order, dtype: float64

In [36]:
# Which cart positions appear most often?
df_orders_products_train["add_to_cart_order"].value_counts().head(20)

add_to_cart_order
1     131209
2     124364
3     116996
4     108963
5     100745
6      91850
7      83142
8      74601
9      66618
10     59401
11     52848
12     46814
13     41431
14     36588
15     32194
16     28363
17     24841
18     21733
19     19014
20     16541
Name: count, dtype: int64

#### Understanding repeated keys in the table

In [37]:
df_orders_products_train["order_id"].is_unique

False

In [38]:
df_orders_products_train["product_id"].is_unique

False

#### Consistency checks for the `order_products_train` table

In [39]:
df_orders_products_train["reordered"].isin([0, 1]).all()

np.True_

In [40]:
df_orders_products_train[["order_id", "product_id"]].isna().sum()

order_id      0
product_id    0
dtype: int64

### products

In [41]:
print(df_products.columns)
print(df_products.shape)
df_products.head()

Index(['product_id', 'product_name', 'aisle_id', 'department_id', 'prices'], dtype='str')
(49693, 5)


,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.80
1,2,All-Seasons Salt,104,13,9.30
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50
4,5,Green Chile Anytime Sauce,5,13,4.30


In [42]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 49693 entries, 0 to 49692
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     49693 non-null  int64  
 1   product_name   49677 non-null  str    
 2   aisle_id       49693 non-null  int64  
 3   department_id  49693 non-null  int64  
 4   prices         49693 non-null  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 3.4 MB


In [43]:
df_products.describe()

,product_id,aisle_id,department_id,prices
count,"49,693.00","49,693.00","49,693.00","49,693.00"
mean,"24,844.35",67.77,11.73,9.99
std,"14,343.72",38.32,5.85,453.52
min,1.00,1.00,1.00,1.00
25%,"12,423.00",35.00,7.00,4.10
50%,"24,845.00",69.00,13.00,7.10
75%,"37,265.00",100.00,17.00,11.20
max,"49,688.00",134.00,21.00,"99,999.00"


In [44]:
df_products.isna().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

#### Analytical questions for products

In [45]:
# How many unique products exist?
df_products["product_id"].nunique()

49686

In [46]:
#  Are product IDs unique?
df_products["product_id"].is_unique

False

In [47]:
# How many unique departments are referenced in the product table?
df_products["department_id"].nunique()

21

In [48]:
# How many unique aisles are referenced in the product table?
df_products["aisle_id"].nunique()

134

In [49]:
# Do some product names repeat?
df_products["product_name"].duplicated().sum()

np.int64(20)

#### Consistency checks for the `products` table

In [50]:
df_products["product_id"].is_unique

False

In [51]:
df_products["department_id"].isna().sum()

np.int64(0)

In [52]:
df_products["aisle_id"].isna().sum()

np.int64(0)

### departments_t

In [53]:
print(df_departments.columns)
print(df_departments.shape)
df_departments.head()

Index(['department_id', 'department'], dtype='str')
(21, 2)


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [54]:
df_departments.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 638.0 bytes


In [55]:
df_departments.isna().sum()

department_id    0
department       0
dtype: int64

#### Analytical questions for departments_table

In [56]:
# How many departments exist?
df_departments["department"].nunique()

21

In [57]:
# Are department_id values unique?
df_departments["department_id"].is_unique

True

### aisles

In [58]:
print(df_aisles.columns)
print(df_aisles.shape)
df_aisles.head()

Index(['aisle_id', 'aisle'], dtype='str')
(134, 2)


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [59]:
df_aisles.info()

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 4.2 KB


In [60]:
df_aisles.isna().sum()

aisle_id    0
aisle       0
dtype: int64

In [61]:
df_aisles.dtypes

aisle_id    int64
aisle         str
dtype: object

#### Analytical questions for aisles

In [62]:
## How many aisles exist?
df_aisles["aisle"].nunique()

134

In [63]:
# Are aisle_id values unique?
df_aisles["aisle_id"].is_unique

True

In [64]:
df_aisles.isna().sum()

aisle_id    0
aisle       0
dtype: int64

### customers 

In [65]:
print(df_customers.columns)
print(df_customers.shape)
df_customers.head()

Index(['user_id', 'First Name', 'Surname', 'Gender', 'STATE', 'Age',
       'date_joined', 'n_dependants', 'fam_status', 'income'],
      dtype='str')
(206209, 10)


,user_id,First Name,Surname,Gender,STATE,Age,date_joined,n_dependants,fam_status,income
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374


In [66]:
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 206209 entries, 0 to 206208
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       206209 non-null  int64
 1   First Name    194950 non-null  str  
 2   Surname       206209 non-null  str  
 3   Gender        206209 non-null  str  
 4   STATE         206209 non-null  str  
 5   Age           206209 non-null  int64
 6   date_joined   206209 non-null  str  
 7   n_dependants  206209 non-null  int64
 8   fam_status    206209 non-null  str  
 9   income        206209 non-null  int64
dtypes: int64(4), str(6)
memory usage: 24.2 MB


In [67]:
df_customers.isna().sum()

user_id             0
First Name      11259
Surname             0
Gender              0
STATE               0
Age                 0
date_joined         0
n_dependants        0
fam_status          0
income              0
dtype: int64

In [68]:
df_customers.describe()

,user_id,Age,n_dependants,income
count,"206,209.00","206,209.00","206,209.00","206,209.00"
mean,"103,105.00",49.50,1.50,"94,632.85"
std,"59,527.56",18.48,1.12,"42,473.79"
min,1.00,18.00,0.00,"25,903.00"
25%,"51,553.00",33.00,0.00,"59,874.00"
50%,"103,105.00",49.00,1.00,"93,547.00"
75%,"154,657.00",66.00,3.00,"124,244.00"
max,"206,209.00",81.00,3.00,"593,901.00"


#### Analytical questions for customers table

In [69]:
df_customers["user_id"].nunique()

206209

### states

In [70]:
print(df_states.columns)
print(df_states.shape)
df_states.head()

Index(['state', 'region', 'division'], dtype='str')
(51, 3)


,state,region,division
0,Connecticut,Northeast,New England
1,Maine,Northeast,New England
2,Massachusetts,Northeast,New England
3,New Hampshire,Northeast,New England
4,Rhode Island,Northeast,New England


In [71]:
df_states.info()

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   state     51 non-null     str  
 1   region    51 non-null     str  
 2   division  51 non-null     str  
dtypes: str(3)
memory usage: 2.7 KB


In [72]:
df_states.isna().sum()

state       0
region      0
division    0
dtype: int64

#### Analytical questions for states

In [73]:
df_states['state'].nunique()

51

In [74]:
df_states['region'].nunique()

4

In [75]:
df_states['division'].nunique()

9

## Merge 

In [76]:
df_products_departments = pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left"
)

df_products_departments.head()

,product_id,product_name,aisle_id,department_id,prices,department
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks
1,2,All-Seasons Salt,104,13,9.30,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry


In [77]:
df_products_departments_check = pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left",
    indicator=True
)

df_products_departments_check.head()

,product_id,product_name,aisle_id,department_id,prices,department,_merge
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks,both
1,2,All-Seasons Salt,104,13,9.30,pantry,both
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages,both
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen,both
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry,both


In [78]:
df_products_departments_check["_merge"].value_counts()

_merge
both          49693
left_only         0
right_only        0
Name: count, dtype: int64

In [79]:
df_products_departments_check[
    df_products_departments_check["_merge"] == "left_only"
].head()

,product_id,product_name,aisle_id,department_id,prices,department,_merge


In [80]:
df_products_enriched = pd.merge(
    df_products_departments,
    df_aisles,
    on="aisle_id",
    how="left"
)

df_products_enriched.head()

,product_id,product_name,aisle_id,department_id,prices,department,aisle
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks,cookies cakes
1,2,All-Seasons Salt,104,13,9.30,pantry,spices seasonings
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages,tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen,frozen meals
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry,marinades meat preparation


In [81]:
df_train_transactions = pd.merge(
    df_orders_train,
    df_orders_products_train,
    on="order_id",
    how="left"
)

df_train_transactions.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,1187899,1,11,4,8,14.00,196,1,1
1,1187899,1,11,4,8,14.00,25133,2,1
2,1187899,1,11,4,8,14.00,38928,3,1
3,1187899,1,11,4,8,14.00,26405,4,1
4,1187899,1,11,4,8,14.00,39657,5,1


In [82]:
df_train_transactions_check = pd.merge(
    df_orders_train,
    df_orders_products_train,
    on="order_id",
    how="left",
    indicator=True
)

df_train_transactions_check.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge
0,1187899,1,11,4,8,14.00,196,1,1,both
1,1187899,1,11,4,8,14.00,25133,2,1,both
2,1187899,1,11,4,8,14.00,38928,3,1,both
3,1187899,1,11,4,8,14.00,26405,4,1,both
4,1187899,1,11,4,8,14.00,39657,5,1,both


In [83]:
df_train_transactions_check["_merge"].value_counts()

_merge
both          1384617
left_only           0
right_only          0
Name: count, dtype: int64

In [84]:
df_train_transactions_check[
    df_train_transactions_check["_merge"] == "left_only"
].head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge


In [85]:
print(df_orders_train.shape)
print(df_orders_products_train.shape)
print(df_train_transactions.shape)

(131209, 6)
(1384617, 4)
(1384617, 9)


In [86]:
df_train_transactions.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'product_id', 'add_to_cart_order',
       'reordered'],
      dtype='str')

In [87]:
df_train_analytic = pd.merge(
    df_train_transactions,
    df_products_enriched,
    on="product_id",
    how="left"
)

df_train_analytic.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,department,aisle
0,1187899,1,11,4,8,14.00,196,1,1,Soda,77.00,7.00,9.00,beverages,soft drinks
1,1187899,1,11,4,8,14.00,25133,2,1,Organic String Cheese,21.00,16.00,8.60,dairy eggs,packaged cheese
2,1187899,1,11,4,8,14.00,38928,3,1,0% Greek Strained Yogurt,120.00,16.00,12.60,dairy eggs,yogurt
3,1187899,1,11,4,8,14.00,26405,4,1,XL Pick-A-Size Paper Towel Rolls,54.00,17.00,1.00,household,paper goods
4,1187899,1,11,4,8,14.00,39657,5,1,Milk Chocolate Almonds,45.00,19.00,6.80,snacks,candy chocolate


In [88]:
df_train_analytic_check = pd.merge(
    df_train_transactions,
    df_products_enriched,
    on="product_id",
    how="left",
    indicator=True
)

df_train_analytic_check["_merge"].value_counts()

_merge
both          1384618
left_only          88
right_only          0
Name: count, dtype: int64

In [89]:
df_train_analytic.info()

<class 'pandas.DataFrame'>
RangeIndex: 1384706 entries, 0 to 1384705
Data columns (total 15 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1384706 non-null  int64  
 1   user_id                 1384706 non-null  int64  
 2   order_number            1384706 non-null  int64  
 3   order_dow               1384706 non-null  int64  
 4   order_hour_of_day       1384706 non-null  int64  
 5   days_since_prior_order  1384706 non-null  float64
 6   product_id              1384706 non-null  int64  
 7   add_to_cart_order       1384706 non-null  int64  
 8   reordered               1384706 non-null  int64  
 9   product_name            1383473 non-null  str    
 10  aisle_id                1384618 non-null  float64
 11  department_id           1384618 non-null  float64
 12  prices                  1384618 non-null  float64
 13  department              1384618 non-null  str    
 14  aisle        

In [90]:
df_train_analytic_check.isna().sum()

order_id                     0
user_id                      0
order_number                 0
order_dow                    0
order_hour_of_day            0
days_since_prior_order       0
product_id                   0
add_to_cart_order            0
reordered                    0
product_name              1233
aisle_id                    88
department_id               88
prices                      88
department                  88
aisle                       88
_merge                       0
dtype: int64

In [91]:
df_train_analytic_check[
    df_train_analytic_check["_merge"] == "left_only"
].head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,department,aisle,_merge
10780,3009052,1629,37,0,13,3.00,6799,24,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
52822,195363,7955,12,6,14,8.00,6799,4,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
59749,831974,9000,11,3,13,20.00,6799,19,0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
64159,206208,9640,16,6,11,12.00,6799,6,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
81585,472316,12168,10,4,12,0.00,6799,12,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [92]:
df_customers = df_customers.rename(columns={"STATE": "state"})

df_customer_states = pd.merge(
    df_customers,
    df_states,
    on="state",
    how="left"
)

print(df_customer_states.columns)
print(df_customer_states.shape)

df_customer_states.head()

Index(['user_id', 'First Name', 'Surname', 'Gender', 'state', 'Age',
       'date_joined', 'n_dependants', 'fam_status', 'income', 'region',
       'division'],
      dtype='str')
(206209, 12)


,user_id,First Name,Surname,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665,Midwest,West North Central
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285,West,Mountain
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568,West,Mountain
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049,Midwest,West North Central
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374,South,South Atlantic


In [93]:
df_customer_states_check = pd.merge(
    df_customers,
    df_states,
    on="state",
    how="left",
    indicator=True
)

df_customer_states_check.head()

,user_id,First Name,Surname,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division,_merge
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665,Midwest,West North Central,both
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285,West,Mountain,both
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568,West,Mountain,both
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049,Midwest,West North Central,both
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374,South,South Atlantic,both


In [94]:
df_customer_states_check['_merge'].value_counts()

_merge
both          206209
left_only          0
right_only         0
Name: count, dtype: int64

In [95]:
df_customer_states_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 206209 entries, 0 to 206208
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   user_id       206209 non-null  int64   
 1   First Name    194950 non-null  str     
 2   Surname       206209 non-null  str     
 3   Gender        206209 non-null  str     
 4   state         206209 non-null  str     
 5   Age           206209 non-null  int64   
 6   date_joined   206209 non-null  str     
 7   n_dependants  206209 non-null  int64   
 8   fam_status    206209 non-null  str     
 9   income        206209 non-null  int64   
 10  region        206209 non-null  str     
 11  division      206209 non-null  str     
 12  _merge        206209 non-null  category
dtypes: category(1), int64(4), str(8)
memory usage: 31.4 MB


In [96]:
df_customer_states_check.describe()

,user_id,Age,n_dependants,income
count,"206,209.00","206,209.00","206,209.00","206,209.00"
mean,"103,105.00",49.50,1.50,"94,632.85"
std,"59,527.56",18.48,1.12,"42,473.79"
min,1.00,18.00,0.00,"25,903.00"
25%,"51,553.00",33.00,0.00,"59,874.00"
50%,"103,105.00",49.00,1.00,"93,547.00"
75%,"154,657.00",66.00,3.00,"124,244.00"
max,"206,209.00",81.00,3.00,"593,901.00"


In [97]:
df_customer_states_check.isna().sum()

user_id             0
First Name      11259
Surname             0
Gender              0
state               0
Age                 0
date_joined         0
n_dependants        0
fam_status          0
income              0
region              0
division            0
_merge              0
dtype: int64

In [98]:
df_analytic = pd.merge(
    df_train_analytic,
    df_customer_states,
    on="user_id",
    how="left"
)

print(df_analytic.columns)
print(df_analytic.shape)
df_analytic.head()

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'product_id', 'add_to_cart_order',
       'reordered', 'product_name', 'aisle_id', 'department_id', 'prices',
       'department', 'aisle', 'First Name', 'Surname', 'Gender', 'state',
       'Age', 'date_joined', 'n_dependants', 'fam_status', 'income', 'region',
       'division'],
      dtype='str')
(1384706, 26)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,...,Surname,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division
0,1187899,1,11,4,8,14.00,196,1,1,Soda,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
1,1187899,1,11,4,8,14.00,25133,2,1,Organic String Cheese,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
2,1187899,1,11,4,8,14.00,38928,3,1,0% Greek Strained Yogurt,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
3,1187899,1,11,4,8,14.00,26405,4,1,XL Pick-A-Size Paper Towel Rolls,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
4,1187899,1,11,4,8,14.00,39657,5,1,Milk Chocolate Almonds,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central


In [99]:
df_analytic.info()

<class 'pandas.DataFrame'>
RangeIndex: 1384706 entries, 0 to 1384705
Data columns (total 26 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1384706 non-null  int64  
 1   user_id                 1384706 non-null  int64  
 2   order_number            1384706 non-null  int64  
 3   order_dow               1384706 non-null  int64  
 4   order_hour_of_day       1384706 non-null  int64  
 5   days_since_prior_order  1384706 non-null  float64
 6   product_id              1384706 non-null  int64  
 7   add_to_cart_order       1384706 non-null  int64  
 8   reordered               1384706 non-null  int64  
 9   product_name            1383473 non-null  str    
 10  aisle_id                1384618 non-null  float64
 11  department_id           1384618 non-null  float64
 12  prices                  1384618 non-null  float64
 13  department              1384618 non-null  str    
 14  aisle        

In [100]:
df_analytic.isna().sum()

order_id                      0
user_id                       0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order        0
product_id                    0
add_to_cart_order             0
reordered                     0
product_name               1233
aisle_id                     88
department_id                88
prices                       88
department                   88
aisle                        88
First Name                75432
Surname                       0
Gender                        0
state                         0
Age                           0
date_joined                   0
n_dependants                  0
fam_status                    0
income                        0
region                        0
division                      0
dtype: int64

## Homework P1:

In [101]:
# Number of products by department
df_products_departments["department"].value_counts()

department
personal care      6565
snacks             6264
pantry             5371
beverages          4365
frozen             4007
dairy eggs         3449
household          3085
canned goods       2092
dry goods pasta    1858
produce            1684
bakery             1516
deli               1322
missing            1258
international      1139
breakfast          1116
babies             1081
alcohol            1056
pets                972
meat seafood        907
other               548
bulk                 38
Name: count, dtype: int64

In [102]:
# Number of products by aisle
df_products_enriched["aisle"].value_counts()

aisle
missing                         1258
candy chocolate                 1246
ice cream ice                   1091
vitamins supplements            1038
yogurt                          1026
                                ... 
frozen juice                      47
baby accessories                  44
packaged produce                  32
bulk grains rice dried goods      26
bulk dried fruits vegetables      12
Name: count, Length: 134, dtype: int64

In [103]:
# aisles per department
df_products_enriched.groupby("department")["aisle"].nunique().sort_values(ascending=False)

department
personal care      17
pantry             12
frozen             11
snacks             11
dairy eggs         10
household          10
beverages           8
meat seafood        7
alcohol             5
canned goods        5
deli                5
dry goods pasta     5
produce             5
bakery              5
breakfast           4
babies              4
international       4
bulk                2
pets                2
other               1
missing             1
Name: aisle, dtype: int64

In [104]:
print(df_products.shape)
print(df_products_departments.shape)
print(df_products_enriched.shape)

(49693, 5)
(49693, 6)
(49693, 7)


## Homework P2:

In [105]:
df_products_enriched["aisle"].value_counts().head(10)

aisle
missing                 1258
candy chocolate         1246
ice cream ice           1091
vitamins supplements    1038
yogurt                  1026
chips pretzels           989
tea                      894
packaged cheese          891
frozen meals             880
cookies cakes            874
Name: count, dtype: int64

In [106]:
pd.crosstab(
    df_products_enriched["department"],
    df_products_enriched["aisle"]
)

aisle,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,beers coolers,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
department,,,,,,,,,,,,,,,,,,,,,
alcohol,0,0,0,0,0,0,0,0,0,387,...,0,0,0,0,0,0,0,0,147,0
babies,0,0,44,132,718,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bakery,0,0,0,0,0,297,0,0,0,0,...,0,0,0,241,0,0,0,0,0,0
beverages,0,0,0,0,0,0,0,0,0,0,...,0,894,0,0,0,0,0,344,0,0
breakfast,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
canned goods,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dairy eggs,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1026
deli,0,0,0,0,0,0,0,0,0,0,...,0,0,159,0,0,0,0,0,0,0


## Homework P3:

In [107]:
# basket size by order
df_train_transactions.groupby("order_id").size().head()

order_id
1      8
36     8
38     9
96     7
98    49
dtype: int64

In [108]:
df_train_transactions.groupby("order_id").size().describe()

count   131,209.00
mean         10.55
std           7.93
min           1.00
25%           5.00
50%           9.00
75%          14.00
max          80.00
dtype: float64

In [109]:
# average basket size by day of week
df_train_transactions.groupby("order_dow").size() / df_train_transactions["order_id"].nunique()

order_dow
0   2.47
1   1.57
2   1.22
3   1.18
4   1.18
5   1.35
6   1.58
dtype: float64

In [110]:
# more careful calculation groups at the order level first and then averages by day.
basket_size_by_order = (
    df_train_transactions
    .groupby(["order_id", "order_dow"])
    .size()
    .reset_index(name="basket_size")
)

basket_size_by_order.groupby("order_dow")["basket_size"].mean()

order_dow
0   11.80
1   10.47
2    9.96
3    9.84
4    9.74
5   10.16
6   10.97
Name: basket_size, dtype: float64

In [111]:
# reorder behavior by hour of day
df_train_transactions.groupby("order_hour_of_day")["reordered"].mean()

order_hour_of_day
0    0.57
1    0.58
2    0.58
3    0.58
4    0.60
5    0.62
6    0.65
7    0.65
8    0.64
9    0.62
10   0.61
11   0.59
12   0.59
13   0.59
14   0.59
15   0.58
16   0.59
17   0.59
18   0.58
19   0.59
20   0.60
21   0.61
22   0.60
23   0.59
Name: reordered, dtype: float64

In [112]:
df_train_transactions["product_id"].isna().sum()

np.int64(0)

In [113]:
df_train_transactions["user_id"].isna().sum()

np.int64(0)

In [114]:
# Does every row now represent an order-product combination?
# A quick way to inspect this is to look at repeated order_id values.
df_train_transactions["order_id"].value_counts().head()

order_id
2813632    80
1395075    80
949182     77
2869702    76
341238     76
Name: count, dtype: int64

## Homework P4:  Questions unlocked after the final merge

In [115]:
df_train_analytic["product_name"].value_counts().head(10)

product_name
Banana                    18726
Bag of Organic Bananas    15480
Organic Strawberries      10894
Organic Baby Spinach       9784
Large Lemon                8135
Organic Avocado            7409
Organic Hass Avocado       7293
Strawberries               6494
Limes                      6033
Organic Raspberries        5546
Name: count, dtype: int64

In [116]:
df_train_analytic["department"].value_counts().head(10)

department
produce            409087
dairy eggs         217051
snacks             118862
beverages          113962
frozen             100426
pantry              81242
bakery              48394
canned goods        46799
deli                44291
dry goods pasta     38713
Name: count, dtype: int64

In [117]:
df_train_analytic.groupby("department")["reordered"].mean().sort_values(ascending=False)

department
dairy eggs        0.67
produce           0.66
beverages         0.66
bakery            0.63
pets              0.63
deli              0.62
alcohol           0.61
meat seafood      0.59
snacks            0.58
bulk              0.58
breakfast         0.57
frozen            0.56
babies            0.54
dry goods pasta   0.49
canned goods      0.49
household         0.43
other             0.39
missing           0.38
international     0.38
pantry            0.36
personal care     0.34
Name: reordered, dtype: float64

In [118]:
pd.crosstab(
    df_train_analytic["department"],
    df_train_analytic["reordered"]
)

reordered,0,1
department,,
alcohol,2203,3399
babies,6857,8084
bakery,17702,30692
beverages,38960,75002
breakfast,12654,16898
bulk,573,786
canned goods,24017,22782
dairy eggs,70549,146502
deli,16924,27367


In [119]:
pd.crosstab(
    df_train_analytic["order_dow"],
    df_train_analytic["reordered"],
    normalize="index"
)

reordered,0,1
order_dow,,
0,0.39,0.61
1,0.40,0.60
2,0.41,0.59
3,0.41,0.59
4,0.41,0.59
5,0.39,0.61
6,0.41,0.59


## Homework 5: Cleaning and saving the final analytical dataset

In [ ]:
df_analytic.drop(columns=['user_id', 'product_id', 'aisle_id', 'department_id'],inplace=True)


In [126]:
print(df_analytic.columns)
print(df_analytic.shape)
df_analytic.head()

Index(['order_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'add_to_cart_order', 'reordered',
       'product_name', 'prices', 'department', 'aisle', 'First Name',
       'Surname', 'Gender', 'state', 'Age', 'date_joined', 'n_dependants',
       'fam_status', 'income', 'region', 'division'],
      dtype='str')
(1384706, 22)


,order_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,product_name,prices,department,...,Surname,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division
0,1187899,11,4,8,14.00,1,1,Soda,9.00,beverages,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
1,1187899,11,4,8,14.00,2,1,Organic String Cheese,8.60,dairy eggs,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
2,1187899,11,4,8,14.00,3,1,0% Greek Strained Yogurt,12.60,dairy eggs,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
3,1187899,11,4,8,14.00,4,1,XL Pick-A-Size Paper Towel Rolls,1.00,household,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
4,1187899,11,4,8,14.00,5,1,Milk Chocolate Almonds,6.80,snacks,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central


In [127]:
df_analytic.info()

<class 'pandas.DataFrame'>
RangeIndex: 1384706 entries, 0 to 1384705
Data columns (total 22 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1384706 non-null  int64  
 1   order_number            1384706 non-null  int64  
 2   order_dow               1384706 non-null  int64  
 3   order_hour_of_day       1384706 non-null  int64  
 4   days_since_prior_order  1384706 non-null  float64
 5   add_to_cart_order       1384706 non-null  int64  
 6   reordered               1384706 non-null  int64  
 7   product_name            1383473 non-null  str    
 8   prices                  1384618 non-null  float64
 9   department              1384618 non-null  str    
 10  aisle                   1384618 non-null  str    
 11  First Name              1309274 non-null  str    
 12  Surname                 1384706 non-null  str    
 13  Gender                  1384706 non-null  str    
 14  state        

In [130]:
df_analytic.describe()

,order_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,prices,Age,n_dependants,income
count,"1,384,706.00","1,384,706.00","1,384,706.00","1,384,706.00","1,384,706.00","1,384,706.00","1,384,706.00","1,384,618.00","1,384,706.00","1,384,706.00","1,384,706.00"
mean,"1,706,297.96",17.09,2.70,13.58,17.07,8.76,0.60,14.12,49.38,1.50,"97,662.28"
std,"989,734.21",16.61,2.17,4.24,10.43,7.42,0.49,680.23,18.44,1.12,"41,991.39"
min,1.00,4.00,0.00,0.00,0.00,1.00,0.00,1.00,18.00,0.00,"25,911.00"
25%,"843,370.00",6.00,1.00,10.00,7.00,3.00,0.00,4.30,33.00,0.00,"64,859.00"
50%,"1,701,880.00",11.00,3.00,14.00,15.00,7.00,1.00,7.40,49.00,1.00,"95,765.00"
75%,"2,568,023.00",21.00,5.00,17.00,30.00,12.00,1.00,11.30,65.00,2.00,"126,817.75"
max,"3,421,070.00",100.00,6.00,23.00,30.00,80.00,1.00,"99,999.00",81.00,3.00,"591,089.00"


In [131]:
df_analytic.to_parquet("../data/processed/instacart.parquet", index = False)

## Download ans save

In [121]:
URL = 'https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/raw/'
l = ['customers', 'states']

for name in l:
    dfname = f"{URL}/{name}.csv"
    print(f"Reading {dfname}...")
    df = pd.read_csv(dfname)
    df.to_csv(f"../data/raw/{name}.csv", index=False)
    

Reading https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/raw//customers.csv...
Reading https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/raw//states.csv...


In [ ]:

URL = 'https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema'

lst = ['customers', 'employees', 'orders', 'products', 'sales']

for name in lst:
    dfname = f"{URL}/{name}.csv"
    print(f"Reading {dfname}...")
    df = pd.read_csv(dfname)
    df.to_csv(f"/Users/marikdeveloper/Desktop/sql-analytics-portfolio-1/data/public_schema/{name}.csv", index=False)
      

Reading https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema/customers.csv...
Reading https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema/employees.csv...
Reading https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema/orders.csv...
Reading https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema/products.csv...
Reading https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/sql_module/data/public_schema/sales.csv...


In [132]:
import pandas as pd
pd.show_versions()


INSTALLED VERSIONS
------------------
commit                : ab90747e3dae0e69b1bdbf083820b8075689b34b
python                : 3.13.5
python-bits           : 64
OS                    : Darwin
OS-release            : 25.3.0
Version               : Darwin Kernel Version 25.3.0: Wed Jan 28 20:55:08 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6020
machine               : arm64
processor             : arm
byteorder             : little
LC_ALL                : None
LANG                  : C.UTF-8
LOCALE                : C.UTF-8

pandas                : 3.0.2
numpy                 : 2.4.4
dateutil              : 2.9.0.post0
pip                   : 26.0.1
Cython                : None
sphinx                : None
IPython               : 9.12.0
adbc-driver-postgresql: None
adbc-driver-sqlite    : None
bs4                   : 4.14.3
bottleneck            : None
fastparquet           : 2026.3.0
fsspec                : 2026.3.0
html5lib              : 1.1
hypothesis            : None
gcsfs     